# SUSHRUTA — Authoritative BraTS Preprocessing & Dataset Split Pipeline

## Executive Summary
This notebook executes the authoritative 2D dataset conversion pipeline for the BraTS MRI dataset.

### Rigorous Single Source of Truth & Zero Leakage Principles:
- **Single Source of Truth**: All patient assignments are strictly driven by `split_manifest.json` (70% train / 15% validation / 15% testing split across 1,251 patients).
- **Patient-Level Isolation**: No patient ID is ever split or reshuffled across splits. Hard assertions enforce zero intersection.
- **Slice Extraction**: Axial FLAIR intensity normalization (non-zero z-scoring), binary tumor mask conversion ({0,1}), 256x256 spatial scaling.
- **Metadata Manifest**: Every single generated slice entry is tracked in `data/processed/brats/metadata.csv` mapping `image_filename`, `mask_filename`, `patient_id`, `split`, `slice_idx`, and `is_tumor`.
- **Verification Statement**: Final confirmation output `"DATASET READY FOR MODEL TRAINING"` upon complete audit passage.

---
## Section 1: Environment Setup & Manifest Single Source of Truth

In [ ]:
import os
import sys
import json
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

# Add project root to sys.path
current_dir = os.path.abspath('')
project_root = current_dir if os.path.exists(os.path.join(current_dir, 'src')) else os.path.dirname(current_dir)
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from src.preprocessing.brats_preprocessing import (
    BraTSPreprocessor,
    generate_full_manifest,
    verify_dataset_integrity
)

# Set fixed seed
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

manifest_path = os.path.join(project_root, 'data', 'processed', 'brats', 'split_manifest.json')
manifest = generate_full_manifest(total_patients=1251, seed=SEED, manifest_path=manifest_path)

print(f"[MANIFEST CONFIRMED] Total Patients: {manifest['total_patients']}")
print(f"  * Train Patients: {manifest['train_patients_count']}")
print(f"  * Val Patients:   {manifest['val_patients_count']}")
print(f"  * Test Patients:  {manifest['test_patients_count']}")

---
## Section 2: Executing Authoritative Preprocessing Pipeline
Processing local BraTS volumes into 2D PNG slices driven strictly by `split_manifest.json` and saving metadata manifest `metadata.csv`.

In [ ]:
raw_dir = os.path.join(project_root, 'data', 'raw', 'brats')
output_dir = os.path.join(project_root, 'data', 'processed', 'brats')

preprocessor = BraTSPreprocessor(
    raw_dir=raw_dir,
    output_dir=output_dir,
    target_shape=(256, 256),
    seed=SEED
)

stats = preprocessor.run(
    train_ratio=0.70,
    val_ratio=0.15,
    test_ratio=0.15,
    non_tumor_ratio=0.20
)

print("[PREPROCESSING COMPLETE]")
print("Processed Patients:", stats['processed_patient_counts'])
print("Slice Counts:", stats['slice_counts'])

---
## Section 3: Metadata Manifest CSV Verification

In [ ]:
csv_path = os.path.join(output_dir, 'metadata.csv')
df_meta = pd.read_csv(csv_path)

print(f"Total metadata records: {len(df_meta):,}")
print("Head of metadata.csv:")
print(df_meta.head())

print("\nMetadata Slices per Split:")
print(df_meta['split'].value_counts())

print("\nMetadata Tumor Slices vs Non-Tumor Slices:")
print(df_meta.groupby(['split', 'is_tumor']).size())

---
## Section 4: Authoritative Dataset Integrity & Patient Isolation Audit

In [ ]:
audit_res = verify_dataset_integrity(output_dir=output_dir)

---
## Section 5: Random Sample Visualization with Patient IDs

In [ ]:
fig, axes = plt.subplots(3, 4, figsize=(16, 12))
rng = random.Random(SEED)

for row_idx, split in enumerate(['train', 'val', 'test']):
    img_dir = os.path.join(output_dir, split, 'images')
    mask_dir = os.path.join(output_dir, split, 'masks')
    files = sorted(os.listdir(img_dir))
    sampled_files = rng.sample(files, 2)
    
    for col_offset, fname in enumerate(sampled_files):
        patient_id = fname.split('_slice_')[0]
        img_arr = np.array(Image.open(os.path.join(img_dir, fname)))
        mask_arr = np.array(Image.open(os.path.join(mask_dir, fname)))
        
        col_img = col_offset * 2
        col_mask = col_img + 1
        
        axes[row_idx, col_img].imshow(np.rot90(img_arr), cmap='gray')
        axes[row_idx, col_img].set_title(f"[{split.upper()}] Image\nPatient: {patient_id}", fontsize=9, fontweight='bold')
        axes[row_idx, col_img].axis('off')
        
        axes[row_idx, col_mask].imshow(np.rot90(mask_arr), cmap='Reds', vmin=0, vmax=1)
        axes[row_idx, col_mask].set_title(f"[{split.upper()}] Mask\nPatient: {patient_id}", fontsize=9, fontweight='bold')
        axes[row_idx, col_mask].axis('off')

plt.suptitle("Authoritative BraTS Dataset Samples Across Splits", fontsize=14, fontweight='bold', y=0.98)
plt.tight_layout()
plt.savefig(os.path.join(project_root, 'results', 'figures', 'authoritative_split_samples.png'), dpi=300, bbox_inches='tight')
plt.close(fig)